# H&E Validation

In [ ]:
abs

In [ ]:
import numpy as np
import pandas as pd
import s3fs
import zarr
import anndata as ad

import matplotlib.pyplot as plt
from matplotlib.patches import Circle

from cloudpathlib import S3Path
from gbmcollaboration.definitions import DATA_PATH_CHUV_FILES, DATA_PATH_UKER_FILES, DATA_PATH_CR_FILES

# Inputs
SAMPLE_ID = "CH_G_196a_vis"
S3_RUN_ROOT = "s3://pa-bi4jtrdsa151p4d3m-jepjxh9qrwgxphe8hd3r6exicahnoeuw1b-s3alias/b5bd8840-cc6e-4de2-a78a-22315e9d1734/v7/without_spotclean"
IMG_KEY = "hires"

PITCH_UM = 100.0
SPOT_DIAM_UM = 55.0
PAD_FRAC = 0.05

# Resolve paths
S3_ZARR_PATH = f"{S3_RUN_ROOT}/SpatialData/{SAMPLE_ID}.zarr"

prefix = SAMPLE_ID[:2]
center_map = {
    "CH": ("CHUV", S3Path(str(DATA_PATH_CHUV_FILES["Cell2location"]))),
    "UK": ("UKER", S3Path(str(DATA_PATH_UKER_FILES["Cell2location"]))),
    "CR": ("CR",   S3Path(str(DATA_PATH_CR_FILES["Cell2location"]))),
}
if prefix not in center_map:
    raise ValueError(f"Unknown center prefix '{prefix}' from sample '{SAMPLE_ID}'")

CENTER_NAME, c2l_base = center_map[prefix]
C2L_H5AD = c2l_base / SAMPLE_ID / "cohort_level_4.h5ad"

print("S3_ZARR_PATH:", S3_ZARR_PATH)
print("CENTER_NAME:", CENTER_NAME)
print("C2L_H5AD:", str(C2L_H5AD))
print("C2L exists:", bool(C2L_H5AD.exists()))

# Load image AnnData from Zarr
s3 = s3fs.S3FileSystem(anon=False)

store = s3.get_mapper(S3_ZARR_PATH)
root = zarr.open_group(store=store, mode="r")
table_grp = root["tables"]["table"] if "table" in root["tables"] else root["tables"][list(root["tables"].group_keys())[0]]
ADATA_IMG = ad.read_zarr(table_grp)

lib = next(iter(ADATA_IMG.uns["spatial"].keys()))
IMG = ADATA_IMG.uns["spatial"][lib]["images"][IMG_KEY]
scalefactors = ADATA_IMG.uns["spatial"][lib]["scalefactors"]
sf_key = "tissue_hires_scalef" if IMG_KEY == "hires" else "tissue_lowres_scalef"
SF = float(scalefactors[sf_key])

# Coordinates and µm scale (pitch-based)
use_obs_pixels = {"pxl_col_in_fullres", "pxl_row_in_fullres"}.issubset(ADATA_IMG.obs.columns)
if use_obs_pixels:
    COORDS_BASE = ADATA_IMG.obs[["pxl_col_in_fullres", "pxl_row_in_fullres"]].to_numpy()
    COORDS_SOURCE = "obs[pxl_col_in_fullres, pxl_row_in_fullres]"
else:
    COORDS_BASE = np.asarray(ADATA_IMG.obsm["spatial"])
    COORDS_SOURCE = "obsm['spatial']"

try:
    from sklearn.neighbors import NearestNeighbors
    nn = NearestNeighbors(n_neighbors=2).fit(COORDS_BASE)
    d, _ = nn.kneighbors(COORDS_BASE)
    nn_dist = d[:, 1]
except Exception:
    from scipy.spatial import cKDTree
    tree = cKDTree(COORDS_BASE)
    d, _ = tree.query(COORDS_BASE, k=2)
    nn_dist = d[:, 1]

MEDIAN_PITCH_PX = float(np.median(nn_dist))

h_px, w_px = IMG.shape[:2]
coords_max = COORDS_BASE.max(axis=0)
COORDS_LOOK_FULLRES = (coords_max[0] > 3 * w_px) or (coords_max[1] > 3 * h_px)

if COORDS_LOOK_FULLRES:
    UM_PER_PX_FULLRES = PITCH_UM / MEDIAN_PITCH_PX
    UM_PER_PX_IMG = UM_PER_PX_FULLRES / SF
    X_UM = COORDS_BASE[:, 0] * UM_PER_PX_FULLRES
    Y_UM = COORDS_BASE[:, 1] * UM_PER_PX_FULLRES
else:
    UM_PER_PX_IMG = PITCH_UM / MEDIAN_PITCH_PX
    X_UM = COORDS_BASE[:, 0] * UM_PER_PX_IMG
    Y_UM = COORDS_BASE[:, 1] * UM_PER_PX_IMG

W_UM = w_px * UM_PER_PX_IMG
H_UM = h_px * UM_PER_PX_IMG
IMG_EXTENT = [0.0, float(W_UM), float(H_UM), 0.0]

span = max(float(X_UM.max() - X_UM.min()), float(Y_UM.max() - Y_UM.min()))
pad_um = PAD_FRAC * span
X_LIM = (float(X_UM.min() - pad_um), float(X_UM.max() + pad_um))
Y_LIM = (float(Y_UM.min() - pad_um), float(Y_UM.max() + pad_um))

R_UM = SPOT_DIAM_UM / 2.0
PATCHES = [Circle((float(X_UM[i]), float(Y_UM[i])), R_UM) for i in range(len(X_UM))]

# Load C2L sample h5ad
h5ad_path_str = str(C2L_H5AD).replace("s3://", "")
with s3.open(h5ad_path_str, "rb") as f:
    adata_c2l = ad.read_h5ad(f)

if "means_cell_abundance_w_sf" not in adata_c2l.obsm:
    raise ValueError("means_cell_abundance_w_sf not found in adata_c2l.obsm")

# Factor names for C2L
factor_names = None
if "mod" in adata_c2l.uns and isinstance(adata_c2l.uns["mod"], dict) and "factor_names" in adata_c2l.uns["mod"]:
    factor_names = adata_c2l.uns["mod"]["factor_names"]
elif "factor_names" in adata_c2l.uns:
    factor_names = adata_c2l.uns["factor_names"]
elif "cell_type_names" in adata_c2l.uns:
    factor_names = adata_c2l.uns["cell_type_names"]

X_means = adata_c2l.obsm["means_cell_abundance_w_sf"]
X_means = X_means.toarray() if hasattr(X_means, "toarray") else np.asarray(X_means)

if factor_names is None:
    factor_names = [f"celltype_{i}" for i in range(X_means.shape[1])]
factor_names = [str(x) for x in factor_names]

# Align rows by obs_names (strict, no transforms)
img_idx = pd.Index(ADATA_IMG.obs_names).astype(str)
c2l_idx = pd.Index(adata_c2l.obs_names).astype(str)
pos = c2l_idx.get_indexer(img_idx)

if np.any(pos < 0):
    missing = img_idx[pos < 0]
    raise ValueError(f"C2L is missing {len(missing)} barcodes from image AnnData. Example: {list(missing[:5])}")

C2L_MEANS = X_means[pos, :]
C2L_FACTOR_NAMES = factor_names

# Fast lookup dict: name -> vector
C2L_MAP = {name: C2L_MEANS[:, j] for j, name in enumerate(C2L_FACTOR_NAMES)}

# Simple diagnostic
print("")
print("Loaded ADATA_IMG:", ADATA_IMG)
print("Loaded adata_c2l:", adata_c2l)
print("COORDS_SOURCE:", COORDS_SOURCE)
print("COORDS_LOOK_FULLRES:", bool(COORDS_LOOK_FULLRES))
print("MEDIAN_PITCH_PX:", MEDIAN_PITCH_PX)
print("UM_PER_PX_IMG:", float(UM_PER_PX_IMG))
print("C2L factors:", len(C2L_FACTOR_NAMES))
print("C2L factor head:", C2L_FACTOR_NAMES[:12])

GENE_CACHE = {}

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection

CELLTYPE = "AC1-like"  # pick from C2L_FACTOR_NAMES
ALPHA = 0.65
CLIP_Q = (1, 99)
FIGSIZE = (8, 8)

if CELLTYPE not in C2L_MAP:
    raise ValueError(f"'{CELLTYPE}' not found. Example options: {C2L_FACTOR_NAMES[:20]}")

v = np.asarray(C2L_MAP[CELLTYPE], dtype=float)

if CLIP_Q is None:
    vmin = float(np.min(v))
    vmax = float(np.max(v))
else:
    vmin = float(np.percentile(v, CLIP_Q[0]))
    vmax = float(np.percentile(v, CLIP_Q[1]))
    if vmax <= vmin:
        vmin = float(np.min(v))
        vmax = float(np.max(v))

pc = PatchCollection(PATCHES, linewidth=0.0, alpha=ALPHA)
pc.set_array(v)
pc.set_clim(vmin, vmax)

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.imshow(IMG, extent=IMG_EXTENT)
ax.add_collection(pc)

ax.set_xlim(X_LIM[0], X_LIM[1])
ax.set_ylim(Y_LIM[1], Y_LIM[0])
ax.set_aspect("equal")
ax.set_xlabel("µm")
ax.set_ylabel("µm")

cbar = fig.colorbar(pc, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label(f"C2L mean abundance: {CELLTYPE}")

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.cm import ScalarMappable

# -------------------------
# Inputs you will change
# -------------------------
MODE = "gene"              # "gene" or "c2l"
KEY = "COL1A1"             # gene name if MODE="gene", or cell type if MODE="c2l"
LAYER = None               # e.g. "log_norm_count" for genes, otherwise None

DISPLAY_DPI = 600          # display-only resolution control
FIGSIZE = (12, 12)           # display-only size control
IMG_INTERPOLATION = "bilinear"  # "none", "nearest", "bilinear", "bicubic"

CMAP = "viridis"
CLIP_Q = (1, 99)           # percentile clipping for color scale, or None

SPOT_FILL_ALPHA = 0.0
SPOT_OUTLINE_ALPHA = 0.6
SPOT_OUTLINE_LW = 0.5

# -------------------------
# Assumes these exist from your load/prep cell:
# ADATA_IMG, IMG, IMG_EXTENT, PATCHES, X_LIM, Y_LIM
# And for MODE="c2l": C2L_MAP, C2L_FACTOR_NAMES
# -------------------------
missing = [name for name in ["IMG", "IMG_EXTENT", "PATCHES", "X_LIM", "Y_LIM"] if name not in globals()]
if missing:
    raise ValueError(f"Missing globals from prep cell: {missing}")

# -------------------------
# Get the vector to plot
# -------------------------
if MODE == "gene":
    if "ADATA_IMG" not in globals():
        raise ValueError("Missing ADATA_IMG. Run the prep/load cell first.")
    if KEY not in ADATA_IMG.var_names:
        raise ValueError(f"Gene '{KEY}' not found in ADATA_IMG.var_names.")
    Xg = ADATA_IMG[:, KEY].layers[LAYER] if LAYER is not None else ADATA_IMG[:, KEY].X
    v = Xg.toarray().ravel() if hasattr(Xg, "toarray") else np.asarray(Xg).ravel()
    label = f"{KEY} expression"
elif MODE == "c2l":
    if "C2L_MAP" not in globals():
        raise ValueError("Missing C2L_MAP. Run the prep/load cell that loads Cell2location first.")
    if KEY not in C2L_MAP:
        ex = C2L_FACTOR_NAMES[:20] if "C2L_FACTOR_NAMES" in globals() else []
        raise ValueError(f"Cell type '{KEY}' not found in C2L_MAP. Example options: {ex}")
    v = np.asarray(C2L_MAP[KEY], dtype=float)
    label = f"C2L mean abundance: {KEY}"
else:
    raise ValueError("MODE must be 'gene' or 'c2l'")

# -------------------------
# Color scaling
# -------------------------
v_plot = v.astype(float)
mask = np.isfinite(v_plot)

if not np.any(mask):
    raise ValueError("All values are non-finite, cannot plot.")

if CLIP_Q is None:
    vmin = float(np.nanmin(v_plot))
    vmax = float(np.nanmax(v_plot))
else:
    vmin = float(np.nanpercentile(v_plot[mask], CLIP_Q[0]))
    vmax = float(np.nanpercentile(v_plot[mask], CLIP_Q[1]))
    if vmax <= vmin:
        vmin = float(np.nanmin(v_plot))
        vmax = float(np.nanmax(v_plot))

norm = plt.Normalize(vmin=vmin, vmax=vmax)
cmap = plt.get_cmap(CMAP)

rgba = cmap(norm(v_plot))
rgba[~mask, :] = (0, 0, 0, 0)  # fully transparent for NaNs

fill_rgba = rgba.copy()
fill_rgba[:, 3] = np.where(mask, SPOT_FILL_ALPHA, 0.0)

edge_rgba = rgba.copy()
edge_rgba[:, 3] = np.where(mask, SPOT_OUTLINE_ALPHA, 0.0)

# Two collections so fill and outline can have different alpha and linewidth,
# while sharing the same color mapping.
pc_fill = PatchCollection(
    PATCHES,
    facecolors=fill_rgba,
    edgecolors="none",
    linewidth=0.0,
    match_original=False,
)
pc_edge = PatchCollection(
    PATCHES,
    facecolors="none",
    edgecolors=edge_rgba,
    linewidth=SPOT_OUTLINE_LW,
    match_original=False,
)

# -------------------------
# Plot
# -------------------------
fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DISPLAY_DPI)

ax.imshow(IMG, extent=IMG_EXTENT, interpolation=IMG_INTERPOLATION)
ax.add_collection(pc_fill)
ax.add_collection(pc_edge)

ax.set_xlim(X_LIM[0], X_LIM[1])
ax.set_ylim(Y_LIM[1], Y_LIM[0])
ax.set_aspect("equal")
ax.set_xlabel("µm")
ax.set_ylabel("µm")

sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label(label)

plt.show()